# Step 1 — Building a News Corpus from the Guardian

Welcome! Over five steps, we are going to build a complete pipeline for
**auditing how well LLMs know what's happening in the world right now**:

| Step | What we build | Output |
|---|---|---|
| **1 (today)** | Scrape fresh news from the Guardian | `data/articles/*.jsonl` |
| 2 | Generate multiple-choice questions from those articles with an LLM | `data/questions/*.jsonl` |
| 3 | LLM-as-judge: vet every question for quality, faithfulness, difficulty | vetted questions |
| 4 | Test LLMs on the quiz: closed-book vs. web search vs. multi-agent debate | predictions |
| 5 | 🏇 **The horse race**: you vs. the LLMs, live | leaderboard |

**Why the Guardian?** Most news APIs give you headlines and links. The
[Guardian Content API](https://open-platform.theguardian.com/) is one of the
few *free* APIs that returns the **full article body text** — exactly what we
need to generate questions in Step 2. It covers every Guardian article back
to 1999, with clean JSON and generous limits.

By the end of this notebook you will understand every line of a
production-grade API collector: one raw request → pagination → rate limits →
call budgets → crash-safe saving → resume → automatic retries.

## 1. Getting a key

Register for a free **developer key** (takes ~2 minutes):
👉 https://open-platform.theguardian.com/access/

Then export it in the terminal you launch Jupyter from:

```bash
export GUARDIAN_API_KEY="your-key-here"
```

**No key yet?** The Guardian provides a shared public key, literally the
string **`test`**, for light experimentation. This notebook falls back to it
automatically, so you can follow along right now and register later.

Two numbers on the free tier drive *every design decision* in this notebook:

> ⏱️ **1 call per second** &nbsp;&nbsp;&nbsp; 📅 **500 calls per day**

Keep them in mind — everything we build (pacing, budgets, resume) exists to
respect these limits while still getting our data reliably.

In [1]:
import json
import os
import time

import requests

API_KEY = os.environ.get("GUARDIAN_API_KEY", "test")
print("Using your own key ✅" if API_KEY != "test" else "Using the shared public 'test' key (fine for the demo)")

Using the shared public 'test' key (fine for the demo)


## 2. One raw API call

Everything starts with a single HTTP GET request to the search endpoint.
`requests` turns the `params` dict into the `?q=...&api-key=...` query string
for us:

In [2]:
BASE = "https://content.guardianapis.com/search"

r = requests.get(BASE, params={"q": "artificial intelligence", "api-key": API_KEY})

print("Status code:", r.status_code)   # 200 = success
print("Actual URL requested:")
print(r.url.replace(API_KEY, "<redacted>"))  # hide my key in the output

Status code: 200
Actual URL requested:
https://content.guardianapis.com/search?q=artificial+intelligence&api-key=<redacted>


## 3. Inspecting the response

The API wraps everything in a `response` envelope. The metadata is as
valuable as the results — `total` and `pages` tell us how much content
matches our search *before* we commit to downloading it:

In [3]:
response = r.json()["response"]

print("Envelope keys:", list(response.keys()), "\n")

for key, value in response.items():
    if key not in ["results"]:
        print(f"{key:15} : {value}")

Envelope keys: ['status', 'userTier', 'total', 'startIndex', 'pageSize', 'currentPage', 'pages', 'orderBy', 'results'] 

status          : ok
userTier        : developer
total           : 108617
startIndex      : 1
pageSize        : 10
currentPage     : 1
pages           : 10862
orderBy         : relevance


In [4]:
# What does one result look like?
print(json.dumps(response["results"][0], indent=2))

{
  "id": "technology/2026/jul/10/apple-sues-openai-trade-secrets",
  "type": "article",
  "sectionId": "technology",
  "sectionName": "Technology",
  "webPublicationDate": "2026-07-10T22:33:21Z",
  "webTitle": "Apple sues OpenAI, alleging artificial intelligence company stole trade secrets",
  "webUrl": "https://www.theguardian.com/technology/2026/jul/10/apple-sues-openai-trade-secrets",
  "apiUrl": "https://content.guardianapis.com/technology/2026/jul/10/apple-sues-openai-trade-secrets",
  "isHosted": false,
  "pillarId": "pillar/news",
  "pillarName": "News"
}


Notice what's **missing**: we got the headline (`webTitle`), URL, section,
and date — but **no article text**. That's the whole reason we're here!

## 4. `show-fields`: asking for the article body

Extra fields are opt-in via the `show-fields` parameter. `bodyText` is the
full article as plain text (HTML stripped — perfect for feeding an LLM):

In [5]:
params = {
    "q": "artificial intelligence",
    "api-key": API_KEY,
    "show-fields": "bodyText,headline,byline,trailText,wordcount",
}
response = requests.get(BASE, params=params).json()["response"]

article = response["results"][0]
fields = article["fields"]

print("Fields keys:", list(fields.keys()), "\n")
print("-" * 40)

print(fields["headline"], "—", fields.get("byline", "(no byline)"))
print(fields["trailText"])
print("-" * 40)

print(f"({fields['wordcount']} words)")
print(fields["bodyText"][:100] + " ...")

Fields keys: ['headline', 'trailText', 'byline', 'wordcount', 'bodyText'] 

----------------------------------------
Apple sues OpenAI, alleging artificial intelligence company stole trade secrets — Dara Kerr
Suit claims OpenAI poached Apple workers, coaxing them to share confidential material in bid to create hardware
----------------------------------------
(447 words)
Apple filed a lawsuit against OpenAI on Friday alleging the artificial intelligence firm stole compa ...


## 5. The full search surface

Keyword search is just the beginning. The API supports rich filtering — these
are exactly the knobs our command-line script will expose later:

| Parameter | Example | What it does |
|---|---|---|
| `q` | `"climate policy"` | Keyword/phrase search ([some boolean logic allowed](https://open-platform.theguardian.com/documentation/)) |
| `section` | `politics`, `technology`, `world` | Restrict to one section |
| `tag` | `environment/climate-crisis` | Restrict to an editorial tag |
| `from-date` / `to-date` | `2026-07-01` | Publication date window |
| `order-by` | `newest`, `oldest`, `relevance` | Result ordering |
| `page-size` | 1–50 | Results per call (default 10, max 50) |
| `page` | 1, 2, 3, … | Which page to fetch |

Let's try a couple:

In [6]:
# Everything the politics desk published in the last week — no keyword at all
response = requests.get(BASE, params={
    "section": "politics",
    "from-date": "2026-07-08",
    "order-by": "newest",
    "page-size": 3,
    "api-key": API_KEY,
}).json()["response"]

print(f"{response['total']:,} politics articles since 2026-07-08. Newest three:")
for a in response["results"]:
    print(" ➡️", a["webPublicationDate"][:10], "—", a["webTitle"])

100 politics articles since 2026-07-08. Newest three:
 ➡️ 2026-07-16 — Andy Burnham to promise to ‘fix the big things’ in first speech as Labour leader
 ➡️ 2026-07-16 — ‘Bizarre choice’: business and Labour puzzle over Shabana Mahmood as future chancellor
 ➡️ 2026-07-16 — Keir Starmer makes Sadiq Khan a peer in the House of Lords


In [7]:
# A tag search: articles tagged with the climate-change editorial tag
response = requests.get(BASE, params={
    "tag": "environment/climate-crisis",
    "page-size": 3,
    "api-key": API_KEY,
}).json()["response"]

print(f"{response['total']:,} articles carry the climate-crisis tag. For example:")
for a in response["results"]:
    print(" •", a["webTitle"])

34,348 articles carry the climate-crisis tag. For example:
 • ‘We are waiting with bated breath’: Super El Niño forecast could make 2027 hottest year on record, BoM says
 • How green is Andy Burnham? Britain’s next PM faces tough climate decisions
 • Food scraps and mushrooms: the closed-loop garden behind the world’s first community-powered sauna


## 6. Flattening results into tidy records

The raw JSON is nested (`article["fields"]["bodyText"]`) and carries keys we
don't need. Downstream steps want **flat, predictable records** — so we
define the *one* place where the API's shape is translated into ours:

In [8]:
def to_record(article):
    """Flatten one raw API result into a tidy, analysis-ready dict."""
    fields = article.get("fields", {})
    return {
        "id": article.get("id"),
        "url": article.get("webUrl"),
        "published": article.get("webPublicationDate"),
        "section": article.get("sectionName"),
        "headline": fields.get("headline"),
        "byline": fields.get("byline"),
        "trail_text": fields.get("trailText"),
        "wordcount": fields.get("wordcount"),
        "body_text": fields.get("bodyText"),
    }

record = to_record(article)   # the AI article from section 4
{k: (str(v)[:70] + "…" if v and len(str(v)) > 70 else v) for k, v in record.items()}

{'id': 'technology/2026/jul/10/apple-sues-openai-trade-secrets',
 'url': 'https://www.theguardian.com/technology/2026/jul/10/apple-sues-openai-t…',
 'published': '2026-07-10T22:33:21Z',
 'section': 'Technology',
 'headline': 'Apple sues OpenAI, alleging artificial intelligence company stole trad…',
 'byline': 'Dara Kerr',
 'trail_text': 'Suit claims OpenAI poached Apple workers, coaxing them to share confid…',
 'wordcount': '447',
 'body_text': 'Apple filed a lawsuit against OpenAI on Friday alleging the artificial…'}

In [9]:
import pandas as pd

# Records are uniform dicts, so a DataFrame is one line away:
params = {"q": "misinformation", "show-fields": "bodyText,headline,byline,trailText,wordcount",
          "page-size": 10, "api-key": API_KEY}
results = requests.get(BASE, params=params).json()["response"]["results"]

df = pd.DataFrame([to_record(a) for a in results])
# df[["published", "section", "headline", "wordcount"]]
df.head()

,id,url,published,section,headline,byline,trail_text,wordcount,body_text
0,uk-news/2026/jul/10/police-warn-against-protes...,https://www.theguardian.com/uk-news/2026/jul/1...,2026-07-10T11:39:38Z,UK news,Police warn about protest misinformation amid ...,Robyn Vinter,Officers say disturbances in city orchestrated...,543,Scottish police have told people to factcheck ...
1,society/2026/may/25/counter-online-misinformat...,https://www.theguardian.com/society/2026/may/2...,2026-05-25T09:00:21Z,Society,Key facts to counter online misinformation abo...,Nicola Davis Science correspondent,Experts say some social media advice could obs...,838,A growing number of women are seeing misleadin...
2,us-news/2026/jun/08/kalshi-polymarket-election...,https://www.theguardian.com/us-news/2026/jun/0...,2026-06-08T23:52:17Z,US news,Kalshi and Polymarket prohibit affiliates from...,Cecilia Nowell,Prediction market apps ask paid content creato...,487,The popular online prediction markets Kalshi a...
3,media/2026/jun/25/disney-brendan-carr-fcc-inve...,https://www.theguardian.com/media/2026/jun/25/...,2026-06-25T19:21:34Z,Media,US media regulator Brendan Carr accuses Disney...,Jeremy Barr in Washington,Disney-owned ABC launched awareness campaign a...,626,"Brendan Carr, the Trump-aligned chairman of th..."
4,society/2026/may/25/misinformation-about-perim...,https://www.theguardian.com/society/2026/may/2...,2026-05-25T09:00:22Z,Society,Misinformation about perimenopause on social m...,Nicola Davis Science correspondent,"Dangers include unintended pregnancies, taking...",790,Misinformation about perimenopause is putting ...


## 7. Pagination — politely

One call returns at most 50 articles. For more, we walk through `page=1, 2,
3, …` — but remember limit #1: **one call per second**. We sleep **1.05 s**
between calls (a small buffer, because our clock and the server's clock never
agree perfectly).

Some mental math on limit #2: `500 calls/day × 50 articles/call = 25,000
articles/day` — more than enough for our tutorial, *if* we don't waste calls.

In [10]:
MIN_INTERVAL = 1.05  # seconds between calls — buffer under the 1/sec limit
PAGE_SIZE = 5        # tiny pages for the demo; use 50 in real runs

all_records = []
for page in [1, 2, 3]:
    if page > 1:
        time.sleep(MIN_INTERVAL)          # politeness first
    response = requests.get(BASE, params={
        "q": "climate policy",
        "show-fields": "bodyText,headline,byline,trailText,wordcount",
        "page-size": PAGE_SIZE,
        "page": page,
        "order-by": "newest",
        "api-key": API_KEY,
    }).json()["response"]
    page_records = [to_record(a) for a in response["results"]]
    all_records.extend(page_records)
    print(f"page {page}/{response['pages']:,}: +{len(page_records)} articles "
          f"(running total: {len(all_records)})")

page 1/79,865: +5 articles (running total: 5)


page 2/79,865: +5 articles (running total: 10)


page 3/79,865: +5 articles (running total: 15)


## 8. Call budgets

A loop over 80,000 pages would happily burn the entire daily allowance while
you're at lunch. A **budget** is just a counter that stops the loop *before*
the API stops you — the difference between a script you can trust overnight
and one you can't:

In [11]:
CALL_BUDGET = 2   # deliberately tiny, to show the stop
calls_made = 0

page = 1
while True:
    if calls_made >= CALL_BUDGET:
        print(f"🛑 Budget of {CALL_BUDGET} calls spent — stopping cleanly.")
        break
    if page > 1:
        time.sleep(MIN_INTERVAL)
    response = requests.get(BASE, params={
        "q": "artificial intelligence", "page-size": 5, "page": page,
        "api-key": API_KEY,
    }).json()["response"]
    calls_made += 1
    print(f"call {calls_made}/{CALL_BUDGET}: fetched page {page} "
          f"({len(response['results'])} articles)")
    page += 1

call 1/2: fetched page 1 (5 articles)


call 2/2: fetched page 2 (5 articles)
🛑 Budget of 2 calls spent — stopping cleanly.


## 9. Saving incrementally with JSONL

Imagine a 400-call collection dying at call 399 — if results only existed in
memory, you would lose *everything* and pay the budget again. So we save
**as we go**.

**JSONL** (JSON Lines) is the ideal format for this: one JSON object per
line, so you can *append* a page at a time. A crash can cost you at most the
current page. (A single big JSON array can't do this — you'd have to rewrite
the whole file every time.)

In [12]:
from pathlib import Path

DEMO_FP = Path("../data/articles/demo_articles.jsonl")
DEMO_FP.parent.mkdir(parents=True, exist_ok=True)
DEMO_FP.unlink(missing_ok=True)   # start the demo fresh

def append_records(records, output_fp):
    """Append records to a JSONL file — one JSON object per line."""
    with open(output_fp, "a", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

append_records(all_records, DEMO_FP)   # the 15 records from section 7
print(f"Wrote {len(all_records)} records. First line of the file:")
print(open(DEMO_FP).readline()[:120] + " …")

Wrote 15 records. First line of the file:
{"id": "us-news/2026/jul/16/trump-tv-address-thursday", "url": "https://www.theguardian.com/us-news/2026/jul/16/trump-tv …


## 10. Resume: never pay for the same article twice

Here's the kill-and-restart story: your collection died halfway. You re-run
the script. Without resume, you re-download (and re-*pay* in API calls for)
everything you already have.

The fix is simple because every Guardian article has a stable `id`: read the
ids already in the output file, then **skip** them as they stream past:

In [13]:
def load_existing_ids(output_fp):
    """Which article ids have we already saved?"""
    if not Path(output_fp).exists():
        return set()
    with open(output_fp, encoding="utf-8") as f:
        return {json.loads(line)["id"] for line in f if line.strip()}

existing = load_existing_ids(DEMO_FP)
print(f"Already saved: {len(existing)} articles\n")

# Re-fetch the SAME page we already saved — watch every article get skipped:
response = requests.get(BASE, params={
    "q": "climate policy", "show-fields": "bodyText,headline",
    "page-size": 5, "page": 1, "order-by": "newest", "api-key": API_KEY,
}).json()["response"]

new = skipped = 0
fresh_records = []
for a in response["results"]:
    record = to_record(a)
    if record["id"] in existing:
        skipped += 1
    else:
        fresh_records.append(record)
        new += 1
append_records(fresh_records, DEMO_FP)
print(f"Re-ran the identical query → {new} new, {skipped} skipped. 💸 saved.")

Already saved: 15 articles



Re-ran the identical query → 0 new, 5 skipped. 💸 saved.


## 11. When things go wrong: retries with exponential backoff

Real collections hit failures. They come in two flavors, and the difference
matters:

| Flavor | Examples | Right response |
|---|---|---|
| **Transient** — try again | `429` rate-limited, `500/502/503` server hiccup, dropped connection | wait, then **retry** with growing delays |
| **Permanent** — trying again is useless | `401/403` bad key, `400` malformed query | **fail immediately** with a clear message |

First, a permanent one — a bad key. No amount of retrying fixes this:

In [14]:
r = requests.get(BASE, params={"q": "test", "api-key": "not-a-real-key"})
print("Status:", r.status_code, "→ permanent error; retrying would be pointless")

Status: 401 → permanent error; retrying would be pointless


For *transient* failures we use the [`tenacity`](https://tenacity.readthedocs.io/)
library: a decorator that re-runs a function with **exponentially growing
waits** (2 s, 4 s, 8 s, …) plus random jitter, and gives up after N attempts.
Here it is on a deliberately flaky function — no API needed:

In [15]:
from tenacity import retry, stop_after_attempt, wait_exponential_jitter

attempts = {"n": 0}

@retry(wait=wait_exponential_jitter(initial=0.2, max=2), stop=stop_after_attempt(4))
def flaky():
    attempts["n"] += 1
    print(f"  attempt {attempts['n']} …", end=" ")
    if attempts["n"] < 3:
        print("💥 ConnectionError")
        raise ConnectionError("simulated network blip")
    print("✅ success")
    return "data!"

flaky()

  attempt 1 … 💥 ConnectionError


  attempt 2 … 💥 ConnectionError


  attempt 3 … ✅ success


'data!'

In production we retry **only** transient errors — the decorator gets a
predicate (`retry_if_exception`) that checks whether the failure is a
network error or a retryable status code (429/5xx). Permanent 4xx errors
raise immediately. You'll see exactly this in the toolkit source.

## 12. The payoff: it all lives in `toolkit.guardian`

Everything we just built by hand is packaged — hardened and tested — in this
repo's local `toolkit` package (installed editable by `uv sync`):

| Notebook section | Toolkit home |
|---|---|
| §2–4 one call + fields | `GuardianClient.fetch_page()` |
| §5 search surface | `fetch_page(...)` keyword arguments |
| §6 flattening | `to_record()` |
| §7 pagination + 1.05 s pacing | `GuardianClient.search()` + `RateLimiter` |
| §8 call budgets | `GuardianClient(daily_budget=...)` → `BudgetExhausted` |
| §9 JSONL appends | `append_records()` |
| §10 resume | `load_existing_ids()` |
| §11 retries | `tenacity` on `GuardianClient._get()` |
| **all of it together** | **`collect()`** |

So the whole notebook becomes one function call:

In [16]:
from toolkit.guardian import collect

summary = collect(
    "artificial intelligence",
    output_fp="../data/articles/demo_articles.jsonl",
    max_articles=10,
    page_size=10,
    from_date="2026-07-01",
)
summary

{'new': 6,
 'skipped': 4,
 'calls_used': 1,
 'total_available': 226,
 'budget_exhausted': False}

Read the source — it's short and it *is* this notebook:
[`toolkit/toolkit/guardian.py`](../toolkit/toolkit/guardian.py)

## 13. The research-grade CLI

For real collections you don't want a notebook — you want a script you can
run on a server, point at a config, and re-run safely. That's
[`scripts/01_collect_guardian_news.py`](../scripts/01_collect_guardian_news.py):
the same toolkit code behind an `argparse` interface exposing everything from
section 5, plus budgets, resume, and logging.

In [17]:
!cd .. && uv run python scripts/01_collect_guardian_news.py --help

usage: 01_collect_guardian_news.py [-h] [--query PHRASE [PHRASE ...]]
                                   [--section SECTION] [--tag TAG]
                                   [--from-date YYYY-MM-DD]
                                   [--to-date YYYY-MM-DD]
                                   [--order-by {newest,oldest,relevance}]
                                   [--show-fields FIELD [FIELD ...]]
                                   [--page-size 1-50]
                                   [--max-articles MAX_ARTICLES]
                                   [--daily-budget DAILY_BUDGET]
                                   [--output OUTPUT] [--no-resume]
                                   [--api-key API_KEY]
                                   [--log-level {DEBUG,INFO,WARNING,ERROR}]
                                   [--log-file LOG_FILE | --create-log-file]

Purpose: Collect Guardian articles into a JSONL file via the Guardian Content
API, with rate limiting, a daily call budget, retries, and resum

A realistic research run:

```bash
uv run python scripts/01_collect_guardian_news.py \
    --query "climate policy" "misinformation" "artificial intelligence" \
    --from-date 2026-07-01 \
    --max-articles 200 \
    --output data/articles/guardian_articles.jsonl
```

Interrupt it, re-run it, run it again tomorrow — resume and the daily budget
make it safe every time.

---

### Next up: Step 2 🧠

We have fresh articles with full body text. In the next notebook, an LLM
reads each article and writes **multiple-choice questions** about it — with
structured JSON outputs, so bad generations can't sneak into our quiz.